In [166]:
import os
from dataset_wo_config_files import  CDCR_Dataset
from sentence_transformers import SentenceTransformer, util
import torch

In [2]:
#Load data
ecb_paths = ['../CDCR/data/ECBplus-prep/output_data/train']
cdcr_train = CDCR_Dataset(name="cdcr_train", data_path=ecb_paths,context_method="hsu")

In [ ]:
def remove_special_token(token_list):
    return [token for token in token_list if token not in ('[E]','[/E]')]

In [ ]:
# Load SBERT model
model = SentenceTransformer('all-mpnet-base-v2')

In [243]:
#Better extract all types and topics from mentions
mention_types = ["entity", "event"]
dict_positive_pairs = {mention_type: {} for mention_type in mention_types}
dict_negative_pairs = {mention_type: {} for mention_type in mention_types}


for topic in cdcr_train.topics:
    for mention_type in mention_types:
        keys_current_topic= cdcr_train.mentions_by_topics[topic]
        #filter by mention type
        mentions_sentences_current_topic= {k: remove_special_token(mention.new_context[1])  for k,mention in cdcr_train.mentions.items() if k in keys_current_topic and mention.type==mention_type}
        if not mentions_sentences_current_topic:
            continue
        #concatenate tokens
        sentences_list = [" ".join(sentence) for sentence in list(mentions_sentences_current_topic.values())]
        print("topic",topic,"mention_type",mention_type,len(sentences_list))
        #Compute similarity
        sentence_embeddings = model.encode(sentences_list, convert_to_tensor=True)
        cosine_scores = util.cos_sim(sentence_embeddings, sentence_embeddings)

        list_of_keys = list(mentions_sentences_current_topic.keys())
        # Get the indices of the lower triangular elements
        indices = torch.tril(torch.ones_like(cosine_scores), diagonal=-1).nonzero()

        # Create the dictionary using the indices
        lower_triangle_dict = {(i.item(), j.item()): cosine_scores[i, j].item() for i, j in indices}

        #classify pairs in: pairs from the same sentence, positives and negatives
  
        positive_pairs = {}
        negative_pairs = {}
        for k,v in lower_triangle_dict.items():
            row, col = k
            mention1 = cdcr_train.mentions[list_of_keys[row]]
            mention2 = cdcr_train.mentions[list_of_keys[col]]
            pair_key = f"{mention1.mention_id}###{mention2.mention_id}"
            #Same sentence
            if mention1.conll_doc_key==mention2.conll_doc_key and mention1.sent_id==mention2.sent_id:
                pass
            elif mention1.coref_chain == mention2.coref_chain:
                positive_pairs[pair_key]=v
            else: 
                negative_pairs[pair_key] = v
        
        # Concatenate positive_pairs and negative_pairs to the respective dictionaries
        dict_positive_pairs[mention_type].update(positive_pairs)
        dict_negative_pairs[mention_type].update(negative_pairs)
        



topic 28 mention_type entity 327
topic 28 mention_type event 176
topic 22 mention_type entity 226
topic 22 mention_type event 134
topic 9 mention_type entity 167
topic 9 mention_type event 53
topic 10 mention_type entity 153
topic 10 mention_type event 45
topic 16 mention_type entity 133
topic 16 mention_type event 60
topic 32 mention_type entity 227
topic 32 mention_type event 108
topic 8 mention_type entity 142
topic 8 mention_type event 96
topic 31 mention_type entity 354
topic 31 mention_type event 119
topic 24 mention_type entity 225
topic 24 mention_type event 99
topic 4 mention_type entity 190
topic 4 mention_type event 87
topic 1 mention_type entity 468
topic 1 mention_type event 287
topic 20 mention_type entity 145
topic 20 mention_type event 92
topic 3 mention_type entity 239
topic 3 mention_type event 97
topic 19 mention_type entity 191
topic 19 mention_type event 128
topic 13 mention_type entity 164
topic 13 mention_type event 102
topic 14 mention_type entity 97
topic 14 me

In [282]:
len(dict_positive_pairs["entity"])

41278

In [258]:
import numpy as np
np.median(list(dict_negative_pairs["entity"].values()))

0.4836481511592865

In [269]:
np.median(list(dict_positive_pairs["entity"].values()))

0.6199440956115723

In [273]:
listica = list(dict_negative_pairs["entity"].values())
listica.sort()

In [281]:
listica2= ([valor for valor in listica if valor >0.70])
print(len(listica2))
np.median(listica2)

128674


0.7775762677192688

In [280]:
print(len(listica2))
np.median(listica2)


203817


0.7300561666488647

In [254]:
list(dict_positive_pairs["entity"].values())

[0.8877928256988525,
 0.8753927946090698,
 0.8701074719429016,
 0.724966287612915,
 0.8117293119430542,
 0.7486209869384766,
 0.8185095191001892,
 0.8177617788314819,
 0.6806105971336365,
 0.7422089576721191,
 0.8514009118080139,
 0.9162635803222656,
 0.8886988759040833,
 0.9105562567710876,
 0.8717761635780334,
 0.89585280418396,
 0.9105562567710876,
 0.8717761635780334,
 0.89585280418396,
 0.8795080184936523,
 0.8745774626731873,
 0.9005433917045593,
 0.904643714427948,
 0.904643714427948,
 0.8795080184936523,
 0.8745774626731873,
 0.9005433917045593,
 0.904643714427948,
 0.904643714427948,
 0.8877928256988525,
 0.8303580284118652,
 0.876587450504303,
 0.9348935484886169,
 0.9348935484886169,
 0.8838180303573608,
 0.8838180303573608,
 0.5753154754638672,
 0.4187967777252197,
 0.5066540241241455,
 0.49377331137657166,
 0.49377331137657166,
 0.5165471434593201,
 0.5165471434593201,
 0.4759162664413452,
 0.8933360576629639,
 0.8629688024520874,
 0.9096025228500366,
 0.886473536491394,
 

In [250]:
len(dict_negative_pairs["entity"])

772660

In [249]:
len(dict_positive_pairs["event"])

15584

In [251]:
len(dict_negative_pairs["event"])

172336

In [264]:
listica.sort()

In [265]:
len(listica)

772660

In [268]:
np.median(listica[-23012:])

0.9032562375068665

In [ ]:
## control for the odd topic, compute combinations

# Validation of duplicate topics in train and test

In [31]:
df.query("mention_id==@repeated_mentions")[["source_path","topic_id","type"]].drop_duplicates()

,source_path,topic_id,type
4960,../CDCR/data/ECBplus-prep/output_data/train,34,entity
8470,../CDCR/data/ECBplus-prep/output_data/train,34,event
10043,../CDCR/data/ECBplus-prep/output_data/val,34,entity
11389,../CDCR/data/ECBplus-prep/output_data/val,34,event


In [35]:
df.query("mention_id==@repeated_mentions").sample(10)

,mention_id,source_path,topic_id,type
10263,34_10ecbplus_HUM17634086926494905_1_6_MWws,../CDCR/data/ECBplus-prep/output_data/val,34,entity
5253,34_9ecbplus_HUM17209079221902940_3_1_ZdUd,../CDCR/data/ECBplus-prep/output_data/train,34,entity
8517,34_1ecbplus_ACT17634157508289507_5_5_B8nh,../CDCR/data/ECBplus-prep/output_data/train,34,event
11413,34_10ecb_ACT17640644449345164_0_8_cDdM,../CDCR/data/ECBplus-prep/output_data/val,34,event
5209,34_3ecbplus_TIM17634315013134476_3_43_4Xyx,../CDCR/data/ECBplus-prep/output_data/train,34,entity
10177,34_13ecb_Singleton_HUMR7zgkEY_1_6_g4Mw,../CDCR/data/ECBplus-prep/output_data/val,34,entity
10240,34_6ecb_Singleton_HUMT9kyijV_1_23_3z2Q,../CDCR/data/ECBplus-prep/output_data/val,34,entity
8529,34_1ecbplus_ACT17634128576760407_4_3_jZRm,../CDCR/data/ECBplus-prep/output_data/train,34,event
10047,34_16ecb_HUM17212524027211372_1_5_K4gb,../CDCR/data/ECBplus-prep/output_data/val,34,entity
10283,34_7ecbplus_HUM17634086926494905_1_0_CKq2,../CDCR/data/ECBplus-prep/output_data/val,34,entity


In [5]:
len(all_mentions)

15491

In [6]:
len(set(all_mentions))

15094

In [3]:
cdcr_test.sentences.keys()

dict_keys(['12/12ecb/1', '12/12ecb/10', '12/12ecb/11', '12/12ecb/12', '12/12ecb/13', '12/12ecb/14', '12/12ecb/15', '12/12ecb/16', '12/12ecb/17', '12/12ecb/18', '12/12ecb/19', '12/12ecb/2', '12/12ecb/3', '12/12ecb/4', '12/12ecb/5', '12/12ecb/6', '12/12ecb/7', '12/12ecb/8', '12/12ecb/9', '12/12ecbplus/1', '12/12ecbplus/10', '12/12ecbplus/11', '12/12ecbplus/2', '12/12ecbplus/3', '12/12ecbplus/4', '12/12ecbplus/5', '12/12ecbplus/6', '12/12ecbplus/7', '12/12ecbplus/8', '12/12ecbplus/9', '18/18ecb/1', '18/18ecb/10', '18/18ecb/11', '18/18ecb/12', '18/18ecb/13', '18/18ecb/14', '18/18ecb/15', '18/18ecb/16', '18/18ecb/2', '18/18ecb/3', '18/18ecb/4', '18/18ecb/5', '18/18ecb/6', '18/18ecb/7', '18/18ecb/8', '18/18ecb/9', '18/18ecbplus/1', '18/18ecbplus/10', '18/18ecbplus/11', '18/18ecbplus/2', '18/18ecbplus/3', '18/18ecbplus/4', '18/18ecbplus/5', '18/18ecbplus/6', '18/18ecbplus/7', '18/18ecbplus/8', '18/18ecbplus/9', '2/2ecb/1', '2/2ecb/11', '2/2ecb/2', '2/2ecb/3', '2/2ecb/4', '2/2ecb/5', '2/2ecb/6

In [5]:
cdcr_test.sentences['12/12ecb/15'][0]

{0: 'MSNBC',
 1: 'is',
 2: 'reporting',
 3: 'that',
 4: 'the',
 5: 'Indian',
 6: 'Navy',
 7: 'claims',
 8: 'they',
 9: 'have',
 10: 'captured',
 11: '23',
 12: 'pirates',
 13: 'in',
 14: 'the',
 15: 'Gulf',
 16: 'of',
 17: 'Aden',
 18: '.'}

In [7]:
cdcr_test.mentions[0].sent_id

0

In [8]:
cdcr_test.mentions[0].doc_id


'10'

In [6]:
import random

In [7]:
valrdm = random.randint(1,1000)
cdcr_test.mentions[valrdm].tokens_str

NameError: name 'cdcr_test' is not defined

In [38]:
valrdm

669

In [39]:
cdcr_test.mentions[valrdm].conll_doc_key

'21/21ecb/3'

In [43]:
cdcr_test.mentions[valrdm].tokens_str

'a store'

In [41]:
cdcr_test.sentences['21/21ecb/3'][cdcr_test.mentions[valrdm].sent_id]

{0: 'The',
 1: 'Hennepin',
 2: 'County',
 3: 'Attorney',
 4: 'filed',
 5: 'criminal',
 6: 'charges',
 7: 'Friday',
 8: 'against',
 9: 'the',
 10: 'man',
 11: 'who',
 12: 'they',
 13: 'say',
 14: 'hit',
 15: 'and',
 16: 'killed',
 17: 'a',
 18: 'woman',
 19: 'walking',
 20: 'out',
 21: 'of',
 22: 'a',
 23: 'store',
 24: '.'}

In [18]:
cdcr_test.mentions[0].tokens_str

'pirates'

In [13]:
cdcr_test.sentences[cdcr_test.mentions[0].conll_doc_key][0]

{0: 'The',
 1: 'Indian',
 2: 'navy',
 3: 'has',
 4: 'captured',
 5: '23',
 6: 'Somalian',
 7: 'pirates',
 8: '.'}

In [20]:
cdcr_test.mentions[0].conll_doc_key in cdcr_test.sentences.keys()

True

In [17]:
the_key = cdcr_test.mentions[100].conll_doc_key
the_sentence= cdcr_test.mentions[100].sent_id

cdcr_test.sentences[the_key][the_sentence]

[(0, 'In'),
 (1, 'another'),
 (2, 'successful'),
 (3, 'anti-piracy'),
 (4, 'operation'),
 (5, ','),
 (6, 'Navy'),
 (7, 'warship'),
 (8, 'on'),
 (9, 'Saturday'),
 (10, 'repulsed'),
 (11, 'an'),
 (12, 'attack'),
 (13, 'on'),
 (14, 'a'),
 (15, 'merchant'),
 (16, 'vessel'),
 (17, 'in'),
 (18, 'the'),
 (19, 'Gulf'),
 (20, 'of'),
 (21, 'Aden'),
 (22, 'and'),
 (23, 'nabbed'),
 (24, '23'),
 (25, 'Somali'),
 (26, 'and'),
 (27, 'Yemeni'),
 (28, 'sea'),
 (29, 'brigands'),
 (30, ','),
 (31, 'in'),
 (32, 'a'),
 (33, 'show'),
 (34, 'of'),
 (35, 'resolve'),
 (36, 'to'),
 (37, 'weed'),
 (38, 'out'),
 (39, 'the'),
 (40, 'menace'),
 (41, 'that'),
 (42, 'affected'),
 (43, 'maritime'),
 (44, 'trade'),
 (45, 'in'),
 (46, 'the'),
 (47, 'region'),
 (48, '.'),
 (0, 'In'),
 (1, 'another'),
 (2, 'successful'),
 (3, 'anti-piracy'),
 (4, 'operation'),
 (5, ','),
 (6, 'Navy'),
 (7, 'warship'),
 (8, 'on'),
 (9, 'Saturday'),
 (10, 'repulsed'),
 (11, 'an'),
 (12, 'attack'),
 (13, 'on'),
 (14, 'a'),
 (15, 'merchant'),

In [11]:
cdcr_test.mentions[100].sent_id

0

In [9]:
cdcr_test.mentions[100].tokens_text

['affected']

In [2]:
def get_sentences_by_split(dataset_folder, split_name):
    docs_path = os.path.join(dataset_folder + split_name + '/', 'dataset.conll')
    documents, _ = read_conll_file(docs_path)
    sentences_dict = make_dict_of_sentences_no_cross(documents)

    return documents, sentences_dict

In [3]:
#ECB+:
common_data_folder = "../CDCR/data/"
dataset_name = 'ECBplus-prep'
dataset_folder = f"{common_data_folder}{dataset_name}/output_data/"
split_name = 'train'
ecbplus_documents_train, ecbplus_sentences_train = get_sentences_by_split(dataset_folder, split_name)

split_name = 'test'
ecbplus_documents_test,ecbplus_sentences_test = get_sentences_by_split(dataset_folder, split_name)

split_name = 'val'
ecbplus_documents_val ,ecbplus_sentences_val = get_sentences_by_split(dataset_folder, split_name)

# ecbplus_sentences = ecbplus_sentences_train + ecbplus_sentences_test + ecbplus_sentences_val


In [4]:
ecbplus_sentences_val['12/12ecb/1'][1]

[(0, 'Responding'),
 (1, 'to'),
 (2, 'a'),
 (3, 'distress'),
 (4, 'call'),
 (5, 'by'),
 (6, 'MV'),
 (7, 'Gibe'),
 (8, ','),
 (9, 'INS'),
 (10, 'Mysore'),
 (11, 'on'),
 (12, 'anti-piracy'),
 (13, 'patrol'),
 (14, 'in'),
 (15, 'the'),
 (16, 'area'),
 (17, ','),
 (18, 'swung'),
 (19, 'into'),
 (20, 'action'),
 (21, 'against'),
 (22, 'the'),
 (23, 'pirates'),
 (24, 'in'),
 (25, 'two'),
 (26, 'boats'),
 (27, ','),
 (28, 'a'),
 (29, 'Defence'),
 (30, 'Ministry'),
 (31, 'release'),
 (32, 'said'),
 (33, '.'),
 (0, 'Responding'),
 (1, 'to'),
 (2, 'a'),
 (3, 'distress'),
 (4, 'call'),
 (5, 'by'),
 (6, 'MV'),
 (7, 'Gibe'),
 (8, ','),
 (9, 'INS'),
 (10, 'Mysore'),
 (11, 'on'),
 (12, 'anti-piracy'),
 (13, 'patrol'),
 (14, 'in'),
 (15, 'the'),
 (16, 'area'),
 (17, ','),
 (18, 'swung'),
 (19, 'into'),
 (20, 'action'),
 (21, 'against'),
 (22, 'the'),
 (23, 'pirates'),
 (24, 'in'),
 (25, 'two'),
 (26, 'boats'),
 (27, ','),
 (28, 'a'),
 (29, 'Defence'),
 (30, 'Ministry'),
 (31, 'release'),
 (32, 'said')

In [5]:
dat = dataset_folder + split_name + "/entity_mentions.json"

In [6]:
dat

'../CDCR/data/ECBplus-prep/output_data/val/entity_mentions.json'

In [15]:
dat

'../CDCR/data/ECBplus-prep/output_data/val/entity_mentions.json'

In [7]:
import json
## Load mentions:
with open(dat, 'r') as f:
    mentions = json.load(f)

In [8]:
mentions[0]["coref_chain"]

'HUM17403498594960941'

In [9]:
new_mention = Mention(mentions[0])

In [13]:
new_mention.create_context(["asd","asa"],"new")

In [14]:
new_mention.new_context

['asd', 'asa']

In [8]:
for k,v in mentions[0].items():
    print(k,type(v))


coref_chain <class 'str'>
mention_ner <class 'str'>
mention_head_pos <class 'str'>
mention_head_lemma <class 'str'>
mention_head <class 'str'>
mention_head_id <class 'int'>
doc_id <class 'str'>
doc <class 'str'>
is_continuous <class 'bool'>
is_singleton <class 'bool'>
mention_id <class 'str'>
mention_type <class 'str'>
mention_full_type <class 'str'>
score <class 'float'>
sent_id <class 'int'>
mention_context <class 'list'>
tokens_number_context <class 'list'>
tokens_number <class 'list'>
tokens_str <class 'str'>
tokens_text <class 'list'>
topic_id <class 'str'>
topic <class 'str'>
subtopic_id <class 'str'>
subtopic <class 'str'>
coref_type <class 'str'>
description <class 'str'>
conll_doc_key <class 'str'>


In [ ]:
["coref_chain","doc_id","doc","mention_id","mention_context",'tokens_number',"tokens_str","tokens_text","topic_id","topic","subtopic_id","subtopic"]

In [15]:
mentions[0]

{'coref_chain': 'HUM17403498594960941',
 'mention_ner': 'O',
 'mention_head_pos': 'NOUN',
 'mention_head_lemma': 'pirate',
 'mention_head': 'pirates',
 'mention_head_id': 7,
 'doc_id': '10',
 'doc': '12_10ecb',
 'is_continuous': True,
 'is_singleton': False,
 'mention_id': '12_10ecb_HUM17403498594960941_0_7_dZMa',
 'mention_type': 'HUM',
 'mention_full_type': 'HUMAN_PART_PER',
 'score': -1.0,
 'sent_id': 0,
 'mention_context': ['The',
  'Indian',
  'navy',
  'has',
  'captured',
  '23',
  'Somalian',
  'pirates',
  '.',
  'The',
  'pirates',
  'were',
  'about',
  'to',
  'board',
  'a',
  'vessel',
  'in',
  'the',
  'Gulf',
  'of',
  'Aden',
  'when',
  'they',
  'were',
  'apprehended',
  '.',
  'Last',
  'month',
  ',',
  'India',
  "'s",
  'navy',
  'drew',
  'criticism',
  'after',
  'sinking',
  'a',
  'Thai',
  'fishing',
  'trawler',
  'that',
  'had',
  'been',
  'commandeered',
  'hours',
  'earlier',
  'by',
  'pirates',
  ',',
  'but',
  'the',
  'navy',
  'says',
  'it',


In [ ]:
'coref_chain', 'mention_ner', 'mention_head_pos', 'mention_head_lemma', 'mention_head', 'mention_head_id', 'doc_id', 
'doc', 'is_continuous', 'is_singleton', 'mention_id', 'mention_type', 'mention_full_type', 'score', 'sent_id', 'mention_context', 
'tokens_number_context', 'tokens_number', 'tokens_str', 'tokens_text', 'topic_id', 'topic', 'subtopic_id', 'subtopic', 'coref_type', 
'description', 'conll_doc_key'